# Texture feature classification

Read precomputed GLCM feature tables, explore texture attributes, and train SVM, random-forest, and PCA/SVM variants. The input feature CSVs are not bundled.

**Research code with separate pretrained artifacts.** Extract the model Release asset at the repository root. No images, labels, feature tables, or saved outputs are included. Read [the execution guide](../docs/EXECUTION.md) and [review notes](../docs/REVIEW_NOTES.md) before running experiments.

Paper: https://doi.org/10.1007/s44196-026-01458-x

## Path configuration
Set the project root below. Dataset requests: naoum1496@gmai.com.

In [ ]:
from pathlib import Path
import os

# Run from the repository root or its notebooks/ directory.
# For another location, set STRAWBERRY_PROJECT_ROOT or edit PROJECT_ROOT.
working_dir = Path.cwd()
default_root = working_dir.parent if working_dir.name == "notebooks" else working_dir
PROJECT_ROOT = Path(os.environ.get("STRAWBERRY_PROJECT_ROOT", str(default_root))).expanduser().resolve()
DATA_DIR = PROJECT_ROOT / "data"
MODEL_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"
FEATURE_DIR = DATA_DIR / "features"
for directory in (MODEL_DIR, MODEL_DIR / "SVM_PCA_GLCM", RESULTS_DIR, FEATURE_DIR):
    directory.mkdir(parents=True, exist_ok=True)
print("Project root:", PROJECT_ROOT)


## 1. Imports and feature-table assembly

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

In [ ]:
import pandas as pd

# Load the individual CSV files of NON_Green_Region
early_df = pd.read_csv(f'{FEATURE_DIR}/Non_Green_Region/Early_Stage_glcm_features_with_labels.csv')
medium_df = pd.read_csv(f'{FEATURE_DIR}/Non_Green_Region/Medium_Stage_glcm_features_with_labels.csv')
severe_df = pd.read_csv(f'{FEATURE_DIR}/Non_Green_Region/Severe_Stage_glcm_features_with_labels.csv')



# Concatenate the dataframes into one
entire_df = pd.concat([early_df, medium_df, severe_df])

# Save the merged dataframe to a new CSV file
entire_df.to_csv(f'{FEATURE_DIR}/Non_Green_Entire_GLCM.csv', index=False)


## 2. Inspect and shuffle the feature table

In [ ]:
import pandas as pd

# Assuming 'Entire GLCM.csv' is your merged file
df = pd.read_csv(f'{FEATURE_DIR}/Non_Green_Entire_GLCM.csv')

df.head()


In [ ]:
# Display a concise summary of the dataframe
df.info()



In [ ]:
# Generate descriptive statistics
df.describe()

In [ ]:
import pandas as pd

# Assuming 'Entire GLCM.csv' is your merged file
df = pd.read_csv(f'{FEATURE_DIR}/Non_Green_Entire_GLCM.csv')
# Shuffle the DataFrame
shuffled_df = df.sample(frac=1).reset_index(drop=True)

# Save the shuffled DataFrame, if needed
shuffled_df.to_csv(f'{FEATURE_DIR}/Non_Green_Shuffled Entire GLCM.csv', index=False)


In [ ]:
df= pd.read_csv(f'{FEATURE_DIR}/Non_Green_Shuffled Entire GLCM.csv')
df.head()

In [ ]:
df.info()

## 3. Explore features and labels

In [ ]:


# Drop the 'image_name' column
df = df.drop('image_name', axis=1)

# Display the updated DataFrame
df.head()


In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
# Class distribution
print(df['Label'].value_counts())



In [ ]:
# Assuming `df` is your dataframe and it has a 'Label' column with the stages as strings
df['Label'] = df['Label'].astype('category')

# Set the categories with the specified order
df['Label'] = df['Label'].cat.set_categories(['Early_Stage', 'Medium_Stage', 'Severe_Stage'], ordered=True)

# Now encode the labels, ensuring that Early_Stage is 0, Medium_Stage is 1, and Severe_Stage is 2
df['Label_encoded'] = df['Label'].cat.codes


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Assuming df is already defined and Label_encoded has been added

# Create a copy of the DataFrame and encode the labels
df_encoded = df.copy()
df_encoded['Label_encoded'] = df_encoded['Label'].cat.codes

# Drop non-numerical columns (in this case, 'Label') before calculating the correlation matrix
df_encoded = df_encoded.drop(columns=['Label'])

# Calculate correlation matrix
corr = df_encoded.corr()

# Plot correlation matrix
plt.figure(figsize=(8, 3))  # Adjusted for better visibility
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Matrix with Encoded Labels')
plt.show()


In [ ]:
# Get the mapping of codes to categories
label_mapping = dict(enumerate(df['Label'].cat.categories))

# Print the mapping
for code, category in label_mapping.items():
    print(f"{code} -> {category}")


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Load your DataFrame
# df = pd.read_csv('path_to_your_data.csv')  # Uncomment this line if you need to load the DataFrame

# List of features you want to compare with the labels
features = ['entropy_d1_a0', 'dissimilarity_d1_a0', 'energy_d1_a0', 'contrast_d1_a0', 'homogeneity_d1_a0', 'correlation_d1_a0']

# Create subplots
rows = 2
cols = 3

fig, axs = plt.subplots(rows, cols, figsize=(15, 8))

for i, feature in enumerate(features):
    row = i // cols
    col = i % cols
    sns.boxplot(x='Label', y=feature, data=df, ax=axs[row, col])
    axs[row, col].set_title(f'Box plot of {feature}')

plt.tight_layout()
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Load your DataFrame
# df = pd.read_csv('path_to_your_data.csv')  # Uncomment this line if you need to load the DataFrame

# List of features you want to compare with the labels
features = ['entropy_d1_a0', 'dissimilarity_d1_a0', 'energy_d1_a0', 'contrast_d1_a0', 'homogeneity_d1_a0', 'correlation_d1_a0']

# Create subplots
rows = 2
cols = 3

fig, axs = plt.subplots(rows, cols, figsize=(15, 8))

for i, feature in enumerate(features):
    row = i // cols
    col = i % cols
    sns.violinplot(x='Label', y=feature, data=df, ax=axs[row, col])
    axs[row, col].set_title(f'Violin plot of {feature}')

plt.tight_layout()
plt.show()

## 4. Prepare features and targets

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score, recall_score, precision_score
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load the dataset
file_path = f'{FEATURE_DIR}/Non_Green_Shuffled Entire GLCM.csv'
data = pd.read_csv(file_path)

# Encode the categorical labels to numbers
data['Label'] = data['Label'].astype('category')
data['Label'] = data['Label'].cat.codes

# Separate the features and the target variable
X = data.drop(['image_name', 'Label'], axis=1)
y = data['Label']



In [ ]:
X.head()

In [ ]:
y.head()

## 5. Split data and fit the scaler

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from joblib import dump

# Split the data into training (80%), testing (10%), and validation (10%)
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.1, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=(1/9), random_state=42, stratify=y_temp)

# Normalize the data: fit on training, transform on training, validation, and test sets
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# Save the scaler for future use
dump(scaler, f'{MODEL_DIR}/SVM_PCA_GLCM/NG_scaler.joblib')


## 6. SVM classifier

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import f1_score, recall_score, precision_score
from sklearn.svm import SVC

# Assuming X_train, X_val, y_train, y_val are already defined and scaled appropriately
# as X_train_scaled, X_val_scaled, y_train, y_val

# Define the SVM classifier
svm_classifier = SVC()

# Define the parameters grid for grid search
param_grid = {
    'C': [0.1, 1, 10, 100],
    'gamma': [1, 0.1, 0.01, 0.001],
    'kernel': ['rbf']
}

# Perform grid search using 5-fold cross validation
grid_search = GridSearchCV(svm_classifier, param_grid, cv=5, scoring='accuracy')

# Fit the grid search to the scaled training data
grid_search.fit(X_train_scaled, y_train)

# Print the best parameters found by grid search
print("Best Parameters:", grid_search.best_params_)

# Get the best SVM model
best_svm_model = grid_search.best_estimator_

# Evaluate the best SVM model on the scaled validation set
y_val_pred = best_svm_model.predict(X_val_scaled)

# Calculate evaluation metrics
f1 = f1_score(y_val, y_val_pred, average='weighted')
recall = recall_score(y_val, y_val_pred, average='weighted')
precision = precision_score(y_val, y_val_pred, average='weighted')

# Print the evaluation metrics
print("Validation Set Evaluation Metrics:")
print("F1 Score:", f1)
print("Recall:", recall)
print("Precision:", precision)


In [ ]:
# Calculate and print the classification report for the validation set
report = classification_report(y_val, y_val_pred)
print("Validation Set Classification Report:")
print(report)

In [ ]:
# Evaluate the best SVM model on the scaled test set
y_test_pred = best_svm_model.predict(X_test_scaled)

# Calculate evaluation metrics on the test set
f1_test = f1_score(y_test, y_test_pred, average='weighted')
recall_test = recall_score(y_test, y_test_pred, average='weighted')
precision_test = precision_score(y_test, y_test_pred, average='weighted')

# Print the evaluation metrics on the test set
print("Test Set Evaluation Metrics:")
print("F1 Score:", f1_test)
print("Recall:", recall_test)
print("Precision:", precision_test)

# Generate and print the classification report for the test set
test_classification_report = classification_report(y_test, y_test_pred)
print("\nTest Classification Report:")
print(test_classification_report)

## 7. Random-forest baseline

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Define the Random Forest classifier
rf_classifier = RandomForestClassifier(n_estimators=100, random_state=42)

# Fit the Random Forest classifier to the training data
rf_classifier.fit(X_train_scaled, y_train)

# Evaluate the Random Forest classifier on the validation set
y_val_pred_rf = rf_classifier.predict(X_val_scaled)

# Calculate evaluation metrics for Random Forest classifier
f1_rf = f1_score(y_val, y_val_pred_rf, average='weighted')
recall_rf = recall_score(y_val, y_val_pred_rf, average='weighted')
precision_rf = precision_score(y_val, y_val_pred_rf, average='weighted')

# Print the evaluation metrics for Random Forest classifier
print("Random Forest Validation Set Evaluation Metrics:")
print("F1 Score:", f1_rf)
print("Recall:", recall_rf)
print("Precision:", precision_rf)
val_classification_report = classification_report(y_val, y_val_pred_rf)
print("\n Validation Classification Report:")
print(val_classification_report)



In [ ]:
from sklearn.metrics import confusion_matrix, accuracy_score
import numpy as np
import pandas as pd

# Compute confusion matrix
cm = confusion_matrix(y_val, y_val_pred_rf)
classes = np.unique(y_val)  # get class labels automatically

# Calculate per-class accuracy
per_class_accuracy = []
for i in range(len(classes)):
    TP = cm[i, i]
    FN = cm[i, :].sum() - TP
    FP = cm[:, i].sum() - TP
    TN = cm.sum() - (TP + FP + FN)
    acc_i = (TP + TN) / cm.sum()
    per_class_accuracy.append(acc_i)

# Calculate overall accuracy
overall_accuracy = accuracy_score(y_val, y_val_pred_rf)

# Display results
print("\nOverall Accuracy:", round(overall_accuracy, 4))
print("\nPer-Class Accuracy:")
for cls, acc in zip(classes, per_class_accuracy):
    print(f"Class {cls}: {acc:.4f}")

# Optional: Pretty display
results_df = pd.DataFrame({
    'Class': classes,
    'Per-Class Accuracy': np.round(per_class_accuracy, 4)
})
print("\nPer-Class Accuracy Table:")
print(results_df)


In [ ]:
from sklearn.metrics import accuracy_score, classification_report

# Evaluate the Random Forest classifier on the test set
y_test_pred_rf = rf_classifier.predict(X_test_scaled)

# Calculate evaluation metrics for the Random Forest classifier on the test set
f1_test_rf = f1_score(y_test, y_test_pred_rf, average='weighted')
recall_test_rf = recall_score(y_test, y_test_pred_rf, average='weighted')
precision_test_rf = precision_score(y_test, y_test_pred_rf, average='weighted')

# Print the evaluation metrics for Random Forest classifier on the test set
print("\nRandom Forest Test Set Evaluation Metrics:")
print("F1 Score:", f1_test_rf)
print("Recall:", recall_test_rf)
print("Precision:", precision_test_rf)

# Generate the classification report for the test set predictions
test_classification_report_rf = classification_report(y_test, y_test_pred_rf)

# Print the classification report for the test set predictions
print("\nTest Set Classification Report:")
print(test_classification_report_rf)


In [ ]:
from sklearn.metrics import confusion_matrix, accuracy_score
import numpy as np
import pandas as pd

# --- Compute confusion matrix ---
cm_test = confusion_matrix(y_test, y_test_pred_rf)
classes = np.unique(y_test)

# --- Calculate per-class accuracy ---
per_class_accuracy_test = []
for i in range(len(classes)):
    TP = cm_test[i, i]
    FN = cm_test[i, :].sum() - TP
    FP = cm_test[:, i].sum() - TP
    TN = cm_test.sum() - (TP + FP + FN)
    acc_i = (TP + TN) / cm_test.sum()
    per_class_accuracy_test.append(acc_i)

# --- Calculate overall accuracy ---
overall_accuracy_test = accuracy_score(y_test, y_test_pred_rf)

# --- Print results ---
print("\nRandom Forest Test Set Accuracy Metrics:")
print("Overall Accuracy:", round(overall_accuracy_test, 4))
print("\nPer-Class Accuracy:")
for cls, acc in zip(classes, per_class_accuracy_test):
    print(f"Class {cls}: {acc:.4f}")

# --- Optional: Show in a nice table ---
results_test_df = pd.DataFrame({
    'Class': classes,
    'Per-Class Accuracy': np.round(per_class_accuracy_test, 4)
})
print("\nPer-Class Accuracy Table:")
print(results_test_df)


## 8. Tuned random forest and export

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier

# Define the Random Forest classifier
rf_classifier = RandomForestClassifier(random_state=42)

# Define the parameter grid for grid search
param_grid = {
    'n_estimators': [50, 100, 150],  # Number of trees in the forest
    'max_depth': [None, 10, 20],      # Maximum depth of the trees
    'min_samples_split': [2, 5, 10],  # Minimum number of samples required to split an internal node
    'min_samples_leaf': [1, 2, 4]     # Minimum number of samples required to be a leaf node
}

# Perform grid search using 5-fold cross-validation
grid_search = GridSearchCV(rf_classifier, param_grid, cv=5, scoring='accuracy', n_jobs=-1)

# Fit grid search to the training data
grid_search.fit(X_train_scaled, y_train)

# Print the best parameters found by grid search
print("Best Parameters:", grid_search.best_params_)

# Get the best Random Forest model
best_rf_model = grid_search.best_estimator_


In [ ]:
from sklearn.metrics import classification_report

# Evaluate the best Random Forest model on the validation set
y_val_pred_rf = best_rf_model.predict(X_val_scaled)

# Generate the classification report for the validation set predictions
val_classification_report = classification_report(y_val, y_val_pred_rf)

# Print the classification report for the validation set predictions
print("Validation Set Classification Report:")
print(val_classification_report)

# Evaluate the best Random Forest model on the test set
y_test_pred_rf = best_rf_model.predict(X_test_scaled)

# Generate the classification report for the test set predictions
test_classification_report = classification_report(y_test, y_test_pred_rf)

# Print the classification report for the test set predictions
print("\nTest Set Classification Report:")
print(test_classification_report)


In [ ]:
from sklearn.metrics import confusion_matrix, accuracy_score
import numpy as np
import pandas as pd

# --- Compute confusion matrix for the test set ---
cm_test = confusion_matrix(y_test, y_test_pred_rf)
classes = np.unique(y_test)

# --- Calculate per-class accuracy ---
per_class_accuracy_test = []
for i in range(len(classes)):
    TP = cm_test[i, i]
    FN = cm_test[i, :].sum() - TP
    FP = cm_test[:, i].sum() - TP
    TN = cm_test.sum() - (TP + FP + FN)
    acc_i = (TP + TN) / cm_test.sum()
    per_class_accuracy_test.append(acc_i)

# --- Calculate overall accuracy ---
overall_accuracy_test = accuracy_score(y_test, y_test_pred_rf)

# --- Print results ---
print("\nRandom Forest Test Set Accuracy Metrics:")
print("Overall Accuracy:", round(overall_accuracy_test, 4))
print("\nPer-Class Accuracy:")
for cls, acc in zip(classes, per_class_accuracy_test):
    print(f"Class {cls}: {acc:.4f}")

# --- Optional: Display neatly in a DataFrame ---
results_test_df = pd.DataFrame({
    'Class': classes,
    'Per-Class Accuracy': np.round(per_class_accuracy_test, 4)
})
print("\nPer-Class Accuracy Table:")
print(results_test_df)


In [ ]:
from joblib import dump

# Specify the file path where you want to save the model
model_file_path = f'{MODEL_DIR}/SVM_PCA_GLCM/NG_random_forest_model.joblib'

# Save the trained Random Forest model to the specified file path
dump(best_rf_model, model_file_path)

print("Model saved successfully.")


## 9. PCA with SVM

In [ ]:
# Apply PCA for dimensionality reduction
pca = PCA(n_components=0.95)  # Keep 95% of the variance
X_train_pca = pca.fit_transform(X_train)
X_val_pca = pca.transform(X_val)
X_test_pca = pca.transform(X_test)



In [ ]:
# Initialize PCA with 0.95 variance retained
pca_95 = PCA(n_components=0.95)

# Fit PCA on the training data
pca_95.fit(X_train)

# Calculate the cumulative sum of explained variance ratio
explained_variance_ratio_95 = np.cumsum(pca_95.explained_variance_ratio_)

# Plot the explained variance
plt.figure(figsize=(5, 3))
plt.plot(explained_variance_ratio_95, marker='o')
plt.xlabel('Number of Components')
plt.ylabel('Cumulative Explained Variance')
plt.title('Explained Variance by Different Principal Components')
plt.grid(True)
plt.show()


In [ ]:
# Plotting the full explained variance to see all the principal components
plt.figure(figsize=(5, 3))
components_range = range(1, len(pca_95.explained_variance_ratio_) + 1)
plt.bar(components_range, pca_95.explained_variance_ratio_, alpha=0.7, align='center',
        label='Individual explained variance')
plt.step(components_range, explained_variance_ratio_95, where='mid',
         label='Cumulative explained variance')
plt.xticks(components_range)
plt.xlim(1, len(components_range))
plt.xlabel('Principal Components')
plt.ylabel('Explained Variance Ratio')
plt.title('Explained Variance Ratio by PCA Components')
plt.legend(loc='best')
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
# Plotting the first two principal components
plt.figure(figsize=(5, 3))
plt.scatter(X_train_pca[:, 0], X_train_pca[:, 1], c=y_train, cmap='viridis', edgecolor='k', s=40)
plt.title('First two principal components after PCA')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.colorbar()
plt.show()


In [ ]:
# Train SVM Classifier
svm = SVC(kernel='rbf', class_weight='balanced', random_state=42)
svm.fit(X_train_pca, y_train)



In [ ]:
# Validate the classifier
y_val_pred = svm.predict(X_val_pca)
val_f1 = f1_score(y_val, y_val_pred, average='weighted')
val_recall = recall_score(y_val, y_val_pred, average='weighted')
val_precision = precision_score(y_val, y_val_pred, average='weighted')
val_classification_report = classification_report(y_val, y_val_pred)




In [ ]:
print(f"Validation F1 Score (Weighted): {val_f1:.3f}")
print(f"Validation Recall (Weighted): {val_recall:.3f}")
print(f"Validation Precision (Weighted): {val_precision:.3f}")
print("\nValidation Classification Report:")
print(val_classification_report)


In [ ]:
# Test the classifier on unseen data
y_test_pred = svm.predict(X_test_pca)
test_classification_report = classification_report(y_test, y_test_pred)



In [ ]:
print("\n Test Classification Report:")
print(test_classification_report)

## 10. Historical PCA/SVM tuning experiment

This original experiment fits a scaler on all samples before splitting. It is retained for traceability, not endorsed as a leakage-free evaluation. See the review notes before using its results.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import classification_report, make_scorer, f1_score
import pandas as pd
# Load the dataset
file_path = f'{FEATURE_DIR}/Non_Green_Shuffled Entire GLCM.csv'
data = pd.read_csv(file_path)

# Encode the categorical labels to numbers
data['Label'] = data['Label'].astype('category')
data['Label'] = data['Label'].cat.codes

# Separate the features and the target variable
X = data.drop(['image_name', 'Label'], axis=1)
y = data['Label']

# Normalize the data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split the data into training (80%), testing (10%), and validation (10%)
X_temp, X_test, y_temp, y_test = train_test_split(X_scaled, y, test_size=0.1, stratify=y, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=1/9, stratify=y_temp, random_state=42)



In [ ]:
# It seems there was an issue with accessing the categorical information, let's manually recreate the mapping
# Since we've already transformed 'Label' into codes, we'll reconstruct the mapping from the 'data' DataFrame
from joblib import dump
# Recreating the DataFrame to ensure 'Label' is in its original form before encoding
data = pd.read_csv(file_path)

# Create a mapping of encoded labels to original labels
original_labels = data['Label'].astype('category').cat.categories
label_mapping = {i: label for i, label in enumerate(original_labels)}

# Save the mapping using joblib
# Now, use the `dump` function to save the label mapping to a file
dump(label_mapping, f'{MODEL_DIR}/SVM_PCA_GLCM/label_mapping.joblib')

# Provide the path to the saved label mapping for downloading
#f'{MODEL_DIR}/SVM_PCA_GLCM'


In [ ]:
# Initialize PCA
pca = PCA(n_components=0.95)

# Initialize SVM
svm = SVC(kernel='rbf', class_weight='balanced', random_state=42)

# Create a pipeline
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', pca),
    ('svm', svm)
])

# Define parameter grid
param_grid = {
    'svm__C': [0.1, 1, 10, 100],
    'svm__gamma': [0.001, 0.01, 0.1, 1]
}



In [ ]:
# Custom scorer for GridSearch
f1_scorer = make_scorer(f1_score, average='weighted')

# Setup GridSearchCV
grid_search = GridSearchCV(pipeline, param_grid, scoring=f1_scorer, cv=5, verbose=2, n_jobs=-1)

# Fit GridSearchCV
grid_search.fit(X_train, y_train)



In [ ]:
# Print best parameters and best score
print("Best parameters found: ", grid_search.best_params_)
print("Best validation score (weighted F1): ", grid_search.best_score_)

# Evaluate on the validation set
y_val_pred = grid_search.predict(X_val)
print("Validation Classification Report:")
print(classification_report(y_val, y_val_pred))

# Save the best model
from joblib import dump
dump(grid_search.best_estimator_, f'{MODEL_DIR}/SVM_PCA_GLCM/best_model.joblib')


In [ ]:
from joblib import load
from sklearn.metrics import accuracy_score

# Load the saved model
best_model = load(f'{MODEL_DIR}/SVM_PCA_GLCM/best_model.joblib')

# Make predictions on the test set
y_test_pred = best_model.predict(X_test)

# Calculate the accuracy of the model on the test set
test_accuracy = accuracy_score(y_test, y_test_pred)

print(f"Test Accuracy: {test_accuracy:.3f}")


In [ ]:
from sklearn.metrics import classification_report

# Generate the classification report for the test set predictions
test_classification_report = classification_report(y_test, y_test_pred)

print("Test Set Classification Report:")
print(test_classification_report)
